# Session 2, Module 09: Dataclasses


This module covers:
- @dataclass decorator basics
- Fields with default values and default_factory
- frozen=True for immutable dataclasses
- field() for advanced configuration
- __post_init__ for validation
- Comparison with regular classes and namedtuples

Data Engineering Context:
Dataclasses are perfect for schema definitions, configuration objects,
and record types in data pipelines. They reduce boilerplate significantly.


In [ ]:
from dataclasses import dataclass, field, asdict, astuple, replace
from typing import Optional
from datetime import datetime

## Basic Dataclass


In [ ]:
print("=== Basic Dataclass ===")


@dataclass
class PipelineConfig:
    """
    Pipeline configuration using dataclass.

    The @dataclass decorator automatically generates:
    - __init__() with all fields as parameters
    - __repr__() for debugging
    - __eq__() for equality comparison
    """

    name: str
    source_table: str
    destination_table: str
    batch_size: int


# Create instance — __init__ generated automatically
config = PipelineConfig(
    name="daily_etl",
    source_table="raw_customers",
    destination_table="dim_customers",
    batch_size=1000,
)

# __repr__ generated automatically
print(f"Config: {config}")

# Access fields as attributes
print(f"Name: {config.name}")
print(f"Batch size: {config.batch_size}")

# __eq__ generated — compares all fields
config2 = PipelineConfig("daily_etl", "raw_customers", "dim_customers", 1000)
print(f"\nconfig == config2: {config == config2}")

# Modify fields (mutable by default)
config.batch_size = 2000
print(f"Modified batch_size: {config.batch_size}")

## Default Values


In [ ]:
print("\n=== Default Values ===")


@dataclass
class JobConfig:
    """Dataclass with default values."""

    name: str
    timeout: int = 300
    retries: int = 3
    enabled: bool = True
    tags: list[str] = field(default_factory=list)  # Mutable default!


# Use defaults
job1 = JobConfig(name="extract")
print(f"Job with defaults: {job1}")

# Override some defaults
job2 = JobConfig(name="transform", timeout=600, tags=["critical", "daily"])
print(f"Job with overrides: {job2}")

IMPORTANT: Mutable defaults must use field(default_factory=...)
This is WRONG and will cause bugs:
@dataclass
class Bad:
    items: list = []  # Shared across all instances!
The correct way (shown above):
items: list = field(default_factory=list)
Each instance gets its own list

In [ ]:
job1.tags.append("new_tag")
print(f"\njob1.tags: {job1.tags}")
print(f"job2.tags: {job2.tags}")  # Not affected!

## Frozen Dataclass — Immutable


In [ ]:
print("\n=== Frozen Dataclass ===")


@dataclass(frozen=True)
class Coordinates:
    """
    Immutable coordinates.

    frozen=True makes the dataclass immutable (hashable).
    """

    latitude: float
    longitude: float


# Create instance
location = Coordinates(40.7128, -74.0060)
print(f"Location: {location}")

# Cannot modify (frozen)
try:
    location.latitude = 0.0
except Exception as e:
    print(f"Cannot modify frozen: {type(e).__name__}")

# Frozen dataclasses are hashable — can use as dict key
locations = {
    Coordinates(40.7128, -74.0060): "New York",
    Coordinates(51.5074, -0.1278): "London",
}
print(f"NYC: {locations[Coordinates(40.7128, -74.0060)]}")

# Can use in sets
unique_locations = {
    Coordinates(40.7128, -74.0060),
    Coordinates(40.7128, -74.0060),  # Duplicate removed
    Coordinates(51.5074, -0.1278),
}
print(f"Unique locations: {len(unique_locations)}")

## Field Configuration


In [ ]:
print("\n=== field() Configuration ===")


@dataclass
class DataRecord:
    """Demonstrates field() options."""

    # Regular field
    id: int

    # Field with default
    status: str = "pending"

    # Field excluded from __repr__
    _internal: str = field(default="secret", repr=False)

    # Field excluded from comparison
    cached_value: Optional[float] = field(default=None, compare=False)

    # Field excluded from __init__ (must have default)
    created_at: datetime = field(default_factory=datetime.now, init=False)

    # Field with custom metadata
    description: str = field(default="", metadata={"max_length": 255})


record = DataRecord(id=1, status="active")
print(f"Record: {record}")  # _internal not shown
print(f"Created at: {record.created_at}")

# Compare ignores cached_value
record2 = DataRecord(id=1, status="active")
record2.cached_value = 999.99
print(f"\nrecord == record2: {record == record2}")  # True! cached_value ignored

# Access field metadata
from dataclasses import fields

print("\nField metadata:")
for f in fields(record):
    if f.metadata:
        print(f"  {f.name}: {f.metadata}")

============================================================
__post_init__ — Validation and Derived Fields
============================================================

In [ ]:
print("\n=== __post_init__ for Validation ===")


@dataclass
class Connection:
    """Demonstrates __post_init__ for validation."""

    host: str
    port: int
    database: str
    connection_string: str = field(init=False)  # Computed

    def __post_init__(self):
        """
        Called automatically after __init__.
        Use for validation and computing derived fields.
        """
        # Validation
        if not self.host:
            raise ValueError("Host cannot be empty")

        if not (1 <= self.port <= 65535):
            raise ValueError(f"Invalid port: {self.port}")

        if not self.database:
            raise ValueError("Database cannot be empty")

        # Compute derived field
        self.connection_string = f"postgresql://{self.host}:{self.port}/{self.database}"


# Valid connection
conn = Connection(host="localhost", port=5432, database="warehouse")
print(f"Connection: {conn}")
print(f"Connection string: {conn.connection_string}")

# Invalid connection
try:
    bad_conn = Connection(host="", port=5432, database="db")
except ValueError as e:
    print(f"\nValidation error: {e}")

## Utility Functions


In [ ]:
print("\n=== Utility Functions ===")


@dataclass
class Customer:
    id: int
    name: str
    email: str
    active: bool = True


customer = Customer(1, "Alice", "alice@example.com")

# asdict() — Convert to dictionary
customer_dict = asdict(customer)
print(f"As dict: {customer_dict}")

# astuple() — Convert to tuple
customer_tuple = astuple(customer)
print(f"As tuple: {customer_tuple}")

# replace() — Create copy with modifications
inactive_customer = replace(customer, active=False)
print(f"Original: {customer}")
print(f"Replaced: {inactive_customer}")

# Create from dict
data = {"id": 2, "name": "Bob", "email": "bob@example.com"}
customer2 = Customer(**data)
print(f"From dict: {customer2}")

## Comparison: Class Vs Namedtuple Vs Dataclass


In [ ]:
print("\n=== Comparison: class vs namedtuple vs dataclass ===")

from collections import namedtuple

# 1. REGULAR CLASS — Most verbose
print("1. Regular Class:")


class PersonClass:
    def __init__(self, name: str, age: int):
        self.name = name
        self.age = age

    def __repr__(self):
        return f"PersonClass(name={self.name!r}, age={self.age!r})"

    def __eq__(self, other):
        if not isinstance(other, PersonClass):
            return NotImplemented
        return self.name == other.name and self.age == other.age


p1 = PersonClass("Alice", 30)
print(f"  {p1}")

# 2. NAMEDTUPLE — Immutable, lightweight
print("\n2. namedtuple:")

PersonNT = namedtuple("PersonNT", ["name", "age"])
p2 = PersonNT("Alice", 30)
print(f"  {p2}")
print(f"  Immutable: Cannot modify after creation")

# 3. DATACLASS — Best of both worlds
print("\n3. Dataclass:")


@dataclass
class PersonDC:
    name: str
    age: int


p3 = PersonDC("Alice", 30)
print(f"  {p3}")
print(f"  Mutable (unless frozen=True)")

print("""
Comparison Summary:
┌─────────────────┬─────────┬────────────┬───────────┐
│ Feature         │ Class   │ namedtuple │ dataclass │
├─────────────────┼─────────┼────────────┼───────────┤
│ Auto __init__   │ No      │ Yes        │ Yes       │
│ Auto __repr__   │ No      │ Yes        │ Yes       │
│ Auto __eq__     │ No      │ Yes        │ Yes       │
│ Mutable         │ Yes     │ No         │ Yes*      │
│ Type hints      │ Manual  │ No         │ Yes       │
│ Default values  │ Manual  │ Limited    │ Yes       │
│ Inheritance     │ Yes     │ Limited    │ Yes       │
│ Validation      │ Manual  │ No         │ __post_init__│
│ Hashable        │ Manual  │ Yes        │ frozen=True │
└─────────────────┴─────────┴────────────┴───────────┘
* dataclass with frozen=True is immutable
""")

## Practical: Schema Definition


In [ ]:
print("=== Practical: Schema Definition ===")


@dataclass(frozen=True)
class Column:
    """Column definition for a table schema."""

    name: str
    data_type: str
    nullable: bool = True
    primary_key: bool = False
    description: str = ""


@dataclass
class TableSchema:
    """Table schema definition."""

    table_name: str
    schema_name: str = "public"
    columns: list[Column] = field(default_factory=list)
    created_at: datetime = field(default_factory=datetime.now, init=False)

    def __post_init__(self):
        if not self.columns:
            raise ValueError("Table must have at least one column")

    @property
    def primary_keys(self) -> list[str]:
        """Get primary key column names."""
        return [c.name for c in self.columns if c.primary_key]

    @property
    def full_name(self) -> str:
        """Get fully qualified table name."""
        return f"{self.schema_name}.{self.table_name}"


# Define a schema
customer_schema = TableSchema(
    table_name="customers",
    schema_name="staging",
    columns=[
        Column("id", "INTEGER", nullable=False, primary_key=True),
        Column("name", "VARCHAR(255)", nullable=False),
        Column("email", "VARCHAR(255)", nullable=True),
        Column("created_at", "TIMESTAMP", nullable=False),
    ],
)

print(f"Schema: {customer_schema.full_name}")
print(f"Primary keys: {customer_schema.primary_keys}")
print(f"Columns:")
for col in customer_schema.columns:
    nullable = "NULL" if col.nullable else "NOT NULL"
    pk = " [PK]" if col.primary_key else ""
    print(f"  {col.name}: {col.data_type} {nullable}{pk}")

# Convert to dict for serialization
print(f"\nAs dict: {asdict(customer_schema)}")

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Basic:
  @dataclass
  class MyClass:
      field1: str
      field2: int = 0

Decorator Options:
  @dataclass(frozen=True)    - Immutable/hashable
  @dataclass(order=True)     - Enable <, >, etc.
  @dataclass(slots=True)     - Memory efficient (3.10+)

Field Options:
  field(default=value)           - Default value
  field(default_factory=list)    - Mutable default
  field(init=False)              - Exclude from __init__
  field(repr=False)              - Exclude from __repr__
  field(compare=False)           - Exclude from comparison

__post_init__:
  - Called after __init__
  - Use for validation
  - Compute derived fields

Utility Functions:
  asdict(obj)         - Convert to dict
  astuple(obj)        - Convert to tuple
  replace(obj, **kw)  - Copy with changes
  fields(obj)         - Get field metadata

When to Use:
  - Data containers (records, configs)
  - Schema definitions
  - API request/response models
  - Anywhere you need a simple class with data
""")